# Project 1 — Building a Search Engine from Scratch
**Course:** Multimedia Information Retrieval & Computer Vision (University of Pisa, A.Y. 2026/2027)

---
## Notebook structure
1. Set-up: package installation with pinned versions (pip install package==x.y.z), Google
Drive mounting, and one configuration cell holding all paths and flags (stemming, stop-word
removal, buffer size, BM25 parameters, ...). Print the runtime characteristics: Python version,
number of CPU cores, RAM.
2. Data: download of the collection, queries, and judgements.
3. Implementation: document processing, indexing, query processing, scoring, and extensions. Longer code may be written to modules with %%writefile from within the notebook and im-
ported.
4. Tests (C5).
5. Indexing runs, printing time, peak memory, and index size.
6. Evaluation and experiments, producing every table and plot of the report.
7. Report (Section 8).

---
## 1. Set-up
This section installs the allowed packages with pinned versions, mounts Google Drive for persistent storage, defines the global configuration cell, and logs the hardware and runtime specifications.

In [ ]:
# Package installation with pinned versions
!pip install -q \
    ir_datasets==0.5.6 \
    ir_measures==0.3.3 \
    PyStemmer==2.2.0.1 \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.2.2 \
    tqdm==4.66.4 \
    psutil==5.9.8 \
    matplotlib==3.8.4 \
    seaborn==0.13.2 \
    cython==3.0.10

In [ ]:
# Environment imports, Google Drive mount, and runtime inspection
# 1.2 Environment imports, Google Drive mount, and runtime inspection
import os
import sys
import time
import shutil
import platform
import resource
from dataclasses import dataclass
from pathlib import Path
import psutil

def get_peak_rss_mb() -> float:
    """Returns peak resident set size (RSS) in MB using resource.getrusage"""
    # On Linux, ru_maxrss is in kilobytes; on macOS, it is in bytes
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    divisor = (1024 ** 2) if sys.platform == "darwin" else 1024
    return usage / divisor

# Mount Google Drive if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive                                      # TODO
    drive.mount("/content/drive", force_remount=False)                  # <----------- WE NEED TO CREATE THE FOLDER --------------
    BASE_LOCAL_DIR = Path("/content")                                   # <-------------------------------------------------------
    BASE_DRIVE_DIR = Path("/content/drive/MyDrive/MIRCV_Project1")      # <-------------------------------------------------------
else:
    # Local fallback if testing outside Colab
    BASE_LOCAL_DIR = Path("./local_workspace")                          # <-------------------------------------------------------
    BASE_DRIVE_DIR = Path("./drive_workspace")                          # <-------------------------------------------------------

BASE_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
BASE_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def print_runtime_characteristics() -> None:
    """Prints Python version, CPU cores, RAM, and disk space"""
    vm = psutil.virtual_memory()
    local_disk = shutil.disk_usage(BASE_LOCAL_DIR)
    drive_disk = shutil.disk_usage(BASE_DRIVE_DIR)
    
    print("=" * 60)
    print("RUNTIME CHARACTERISTICS")
    print("=" * 60)
    print(f"Platform       : {platform.platform()}")
    print(f"Python Version : {sys.version.split()[0]} ({platform.python_implementation()})")
    print(f"CPU Cores      : {os.cpu_count()} logical cores ({psutil.cpu_count(logical=False)} physical)")
    print(f"Total RAM      : {vm.total / (1024**3):.2f} GB (Available: {vm.available / (1024**3):.2f} GB)")
    print(f"Process Peak   : {get_peak_rss_mb():.2f} MB")
    print(f"Local Disk     : {local_disk.free / (1024**3):.2f} GB free / {local_disk.total / (1024**3):.2f} GB total ({BASE_LOCAL_DIR})")
    print(f"Drive Disk     : {drive_disk.free / (1024**3):.2f} GB free / {drive_disk.total / (1024**3):.2f} GB total ({BASE_DRIVE_DIR})")
    print("=" * 60)

print_runtime_characteristics()

In [ ]:
# Global Configuration Cell
@dataclass
class EngineConfig:
    # --- Execution and dataset mode ---
    use_dev_subset: bool = False          # True = use first 100,000 passages for fast debugging; False = full 8.8M collection
    dev_subset_size: int = 100_000        # Number of lines in the development subset
    force_reindex: bool = False           # If False, skips indexing when the final index already exists on Drive
    
    # --- Document and query text processing flags ---
    use_stopwords: bool = True            # Switchable stop-word removal flag
    use_stemming: bool = True             # Switchable Snowball/Porter stemming flag
    unicode_normalisation: str = "NFKC"   # Optional Unicode canonicalisation ("NFKC" or None)
    
    # --- Memory-bounded SPIMI indexing parameters ---
    # Max number of (docid, tf) postings held in RAM before flushing an intermediate run to disk.
    max_buffer_postings: int = 15_000_000 
    cleanup_intermediate_runs: bool = True # Delete temporary block runs once final index is merged (saves Drive/disk space)
    
    # --- Query processing and scoring parameters ---
    bm25_k1: float = 0.9                  # Mandatory default k1 = 0.9
    bm25_b: float = 0.4                   # Mandatory default b = 0.4
    default_top_k: int = 1000             # Default number of retrieved documents for TREC runs
    
    # --- Extensions configuration --- TODO: choose Extensions
    # E1: "none" (raw uint32), "vbyte" (d-gaps + Variable-Byte), "text" (human-readable debug format)
    compression_format: str = "none"      
    # E2: Skip pointers block size (0 = disabled, >0 = block size or sqrt(df) if -1)
    skip_block_size: int = 128            
    # E2: Dynamic pruning strategy for disjunctive queries: "none", "maxscore", or "wand"
    pruning_strategy: str = "none"        
    # E3: Traversal strategy: "daat" (Document-At-A-Time) or "taat" (Term-At-A-Time)
    traversal_strategy: str = "daat"      
    
    # --- Storage paths ---
    local_dir: Path = BASE_LOCAL_DIR
    drive_dir: Path = BASE_DRIVE_DIR

    @property
    def collection_archive_path(self) -> Path:
        return self.local_dir / "collection.tar.gz"

    @property
    def full_collection_tsv_path(self) -> Path:
        return self.local_dir / "collection.tsv"

    @property
    def dev_collection_tsv_path(self) -> Path:
        return self.local_dir / f"collection_dev_{self.dev_subset_size}.tsv"

    @property
    def active_collection_path(self) -> Path:
        return self.dev_collection_tsv_path if self.use_dev_subset else self.full_collection_tsv_path

    def get_index_name(self, stopwords: bool = None, stemming: bool = None, compression: str = None) -> str:
        sw = self.use_stopwords if stopwords is None else stopwords
        st = self.use_stemming if stemming is None else stemming
        comp = self.compression_format if compression is None else compression
        scope = f"dev{self.dev_subset_size}" if self.use_dev_subset else "full"
        return f"index_{scope}_sw{int(sw)}_stem{int(st)}_{comp}"

    def get_drive_index_dir(self, **kwargs) -> Path:
        return self.drive_dir / "indices" / self.get_index_name(**kwargs)

    def get_local_index_dir(self, **kwargs) -> Path:
        return self.local_dir / "indices" / self.get_index_name(**kwargs)

    @property
    def runs_dir(self) -> Path:
        path = self.drive_dir / "runs"
        path.mkdir(parents=True, exist_ok=True)
        return path

CONFIG = EngineConfig()
print(f"Active configuration -> Index tag: {CONFIG.get_index_name()}")
print(f"Active collection    -> {CONFIG.active_collection_path}")

---
## 2. Data
This section downloads and extracts the MS MARCO passage collection (`collection.tsv`, 8,841,823 passages) directly onto the local runtime disk (`/content`) to preserve Google Drive quota and maximise I/O throughput. It also creates the 100,000-passage development subset and loads the three standard evaluation query sets and relevance judgements via `ir_datasets`.

In [ ]:
# Download and extract MS MARCO passage collection (idempotent)
import tarfile
from itertools import islice

# Official MS MARCO Passage Ranking collection URL (with mirror fallback)
MSMARCO_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
    "https://msmarco.blob.core.windows.net/msmarcoranking/collection.tar.gz"
]

def prepare_collection(cfg: EngineConfig) -> None:
    """Downloads collection.tar.gz, extracts collection.tsv, and builds the 100k dev subset."""
    # Extract or download full collection.tsv
    if cfg.full_collection_tsv_path.exists() and cfg.full_collection_tsv_path.stat().st_size > 2_900_000_000:
        print(f"Full collection already extracted at {cfg.full_collection_tsv_path} "
              f"({cfg.full_collection_tsv_path.stat().st_size / (1024**3):.2f} GB)")
    else:
        if not cfg.collection_archive_path.exists():
            print("Downloading collection.tar.gz (1 GB)...")
            downloaded = False
            for url in MSMARCO_URLS:
                exit_code = os.system(f'wget -q --show-progress -O "{cfg.collection_archive_path}" "{url}"')
                if exit_code == 0 and cfg.collection_archive_path.exists() and cfg.collection_archive_path.stat().st_size > 900_000_000:
                    downloaded = True
                    break
                print(f"Warning: download from {url} failed, trying fallback...")
            if not downloaded:
                raise RuntimeError("Failed to download collection.tar.gz from all mirrors.")
        
        print("Extracting collection.tsv (3 GB) to local disk...")
        t0 = time.perf_counter()
        with tarfile.open(cfg.collection_archive_path, "r:gz") as tar:
            tar.extract("collection.tsv", path=cfg.local_dir)
        print(f"Extraction completed in {time.perf_counter() - t0:.1f}s.")
        
        # Remove the compressed archive after extraction to free 1 GB on /content
        if cfg.collection_archive_path.exists():
            cfg.collection_archive_path.unlink()
            print("Removed temporary collection.tar.gz to free local disk space.")

    # 2. Build the 100,000-line development subset
    if cfg.dev_collection_tsv_path.exists():
        print(f"Development subset already exists at {cfg.dev_collection_tsv_path}")
    else:
        print(f"Creating development subset ({cfg.dev_subset_size:,} lines)...")
        with open(cfg.full_collection_tsv_path, "rb") as src, \
             open(cfg.dev_collection_tsv_path, "wb") as dst:
            for line in islice(src, cfg.dev_subset_size):
                dst.write(line)
        print(f"Development subset written to {cfg.dev_collection_tsv_path} "
              f"({cfg.dev_collection_tsv_path.stat().st_size / (1024**2):.2f} MB)")

prepare_collection(CONFIG)

# Preview the first 3 lines of the collection
print("\nSample passages from collection.tsv:")
with open(CONFIG.active_collection_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(3):
        pid_str, text_str = f.readline().rstrip("\r\n").split("\t", 1)
        print(f"  [Line {i}] pid={pid_str} | text={text_str[:90]}...")

In [ ]:
# Load test queries and relevance judgements (qrels) via ir_datasets
import ir_datasets
from ir_measures import Qrel

QUERY_SETS_SPECS = {
    "dl19": {
        "ir_id": "msmarco-passage/trec-dl-2019/judged",
        "expected_queries": 43,
        "description": "TREC DL 2019 (graded, 0-3)"
    },
    "dl20": {
        "ir_id": "msmarco-passage/trec-dl-2020/judged",
        "expected_queries": 54,
        "description": "TREC DL 2020 (graded, 0-3)"
    },
    "dev_small": {
        "ir_id": "msmarco-passage/dev/small",
        "expected_queries": 6980,
        "description": "MS MARCO dev (small) (binary)"
    }
}

@dataclass
class EvaluationDataset:
    name: str
    ir_id: str
    description: str
    queries: dict[str, str]          # qid -> raw query text
    qrels: list[Qrel]                # list of ir_measures.Qrel(query_id, doc_id, relevance)
    judged_qids: list[str]           # sorted list of judged query IDs for exact averaging & paired t-test

def load_evaluation_datasets() -> dict[str, EvaluationDataset]:
    """Loads all three benchmark query sets and their relevance judgements into memory."""
    loaded = {}
    print(f"{'Dataset Key':<12} | {'ir_datasets Identifier':<36} | {'Queries':>8} | {'Judged QIDs':>11} | {'Qrels':>8}")
    print("-" * 88)
    
    for key, spec in QUERY_SETS_SPECS.items():
        ds = ir_datasets.load(spec["ir_id"])
        
        # Load queries preserving order
        queries = {str(q.query_id): str(q.text) for q in ds.queries_iter()}
        
        # Load relevance judgements in ir_measures format
        qrels = [
            Qrel(query_id=str(r.query_id), doc_id=str(r.doc_id), relevance=int(r.relevance))
            for r in ds.qrels_iter()
        ]
        
        # Identify all unique judged query IDs (crucial for Section 7.2 zero-result handling)
        judged_qids = sorted({q.query_id for q in qrels})
        
        # Sanity check against Project 1 Specifications (Section 4.2)
        assert len(queries) == spec["expected_queries"], (
            f"Expected {spec['expected_queries']} queries for {key}, got {len(queries)}"
        )
        
        loaded[key] = EvaluationDataset(
            name=key,
            ir_id=spec["ir_id"],
            description=spec["description"],
            queries=queries,
            qrels=qrels,
            judged_qids=judged_qids
        )
        print(f"{key:<12} | {spec['ir_id']:<36} | {len(queries):>8,d} | {len(judged_qids):>11,d} | {len(qrels):>8,d}")
        
    return loaded

BENCHMARKS = load_evaluation_datasets()